# Hazard segmentation and deployment cost

Use original RGB and each VLM's original phrases with one pinned official SAM3 image model. The four conditions are the two VLM phrase lists, reference-present diagnostics, and the fixed 16-concept policy. Freeze 20 test frames (10 per source) and five development warmups before examining predictions. CPU fixtures are synthetic checks; GPU cost and the 6 decimal GB VLM budget remain unmeasured. See [segmentation usage](../docs/hazard_segmentation.md), [benchmark usage](../docs/hazard_benchmark.md), and the [frozen contract](../docs/hazard_interfaces.md).


In [ ]:
from pathlib import Path
from copy import deepcopy
import json
import os
import sys

# Optional locations may be absolute or relative to VLM_evaluation.
REPO_ROOT_OVERRIDE = None
DATA_ROOT_OVERRIDE = None
RUN_ROOT_OVERRIDE = None
CACHE_ROOT_OVERRIDE = None
RUN_NAME = "hazard_prompt_v1"
CUDA_DEVICE = "cuda:0"  # Exactly one explicit CUDA device.
SAM_SETTINGS_OVERRIDES = {}  # checkpoint_path: POSIX-relative cache path; code_source_root: local pinned checkout.

# Independent explicit actions. Defaults never prepare, load, download, or run models.
PREPARE_SELECTION = False
RUN_FIXTURE = False
RUN_SEGMENTATION = False
RUN_PROFILES = False
ALLOW_DOWNLOADS = False
ENABLE_COMBINED = False
PROFILE_MODEL_KEY = "qwen3_vl_4b"
PROFILE_CONDITION_KEY = None  # SAM override: fixed_policy/reference_present; None uses PROFILE_MODEL_KEY.

configured_repo = REPO_ROOT_OVERRIDE or os.environ.get("TRAVERSABILITY_REPO_ROOT")
search_roots = ([Path(configured_repo).expanduser().resolve()] if configured_repo
                else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
candidates = [path for base in search_roots for path in (base, base / "VLM_evaluation")]
COMPONENT_ROOT = next((path for path in candidates
                       if (path / "docs/hazard_interfaces.md").is_file()), None)
if COMPONENT_ROOT is None:
    raise RuntimeError("Set TRAVERSABILITY_REPO_ROOT to VLM_evaluation or its parent checkout.")

def configured_path(override, variable, default):
    path = Path(override or os.environ.get(variable) or default).expanduser()
    return (path if path.is_absolute() else COMPONENT_ROOT / path).resolve()

DATA_ROOT = configured_path(DATA_ROOT_OVERRIDE, "TRAVERSABILITY_DATA_ROOT", "data")
RUN_ROOT = configured_path(RUN_ROOT_OVERRIDE, "TRAVERSABILITY_RUN_ROOT", "runs")
CACHE_ROOT = configured_path(CACHE_ROOT_OVERRIDE, "TRAVERSABILITY_CACHE_ROOT", ".cache")
RUN_DIR = RUN_ROOT / RUN_NAME
POLICY_PATH = COMPONENT_ROOT / "configs/hazards/policy.json"
source_path = str(COMPONENT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

# Preserve existing summaries when rerunning this configuration cell.
if "condition_summary" not in globals():
    condition_summary = None
if "profiles" not in globals():
    profiles = []
if "fixture_run_dir" not in globals():
    fixture_run_dir = None
print({"component": str(COMPONENT_ROOT), "run": str(RUN_DIR),
       "data": str(DATA_ROOT), "cache": str(CACHE_ROOT), "device": CUDA_DEVICE})


## Read-only configuration

Use an isolated Python 3.12+ environment. Install the segmentation requirements and compatible VLM dependencies during explicit setup. `ALLOW_DOWNLOADS` only permits downloads inside enabled real actions; it never starts a download by itself. Use local/pre-cached checkpoint and pinned source settings in `SAM_SETTINGS_OVERRIDES` for offline runs. Package imports do not load model libraries or weights.


In [ ]:
from traversability_hazard_segmentation.common import load_config, resolve_config

segmentation_config = load_config(COMPONENT_ROOT / "configs/hazards/segmentation.json")
benchmark_config = load_config(COMPONENT_ROOT / "configs/hazards/benchmark.json")
runtime_paths = {"run_dir": str(RUN_DIR), "data_root": str(DATA_ROOT),
                 "cache_root": str(CACHE_ROOT), "policy_path": str(POLICY_PATH),
                 "component_root": str(COMPONENT_ROOT)}
segmentation_config.update(runtime_paths)
benchmark_config.update(runtime_paths)
sam_settings = deepcopy(segmentation_config.get("sam_settings", segmentation_config.get("settings", {})))
sam_settings.update(SAM_SETTINGS_OVERRIDES)
sam_settings.update(device=CUDA_DEVICE, allow_downloads=ALLOW_DOWNLOADS,
                    local_files_only=not ALLOW_DOWNLOADS,
                    enable_model_loading=RUN_SEGMENTATION or RUN_PROFILES,
                    cache_root=str(CACHE_ROOT), policy_path=str(POLICY_PATH))
segmentation_config.update(sam_settings=sam_settings, fixture=False,
                           enable_real_run=RUN_SEGMENTATION)
benchmark_config.update(sam_settings=deepcopy(sam_settings), fixture=False,
                        enable_real_run=RUN_PROFILES, enable_combined=ENABLE_COMBINED)
vlm_settings = deepcopy(benchmark_config.get("vlm_settings", {}))
vlm_settings.update(device=CUDA_DEVICE, local_files_only=not ALLOW_DOWNLOADS,
                    allow_downloads=ALLOW_DOWNLOADS)
benchmark_config["vlm_settings"] = vlm_settings
resolved_paths = resolve_config(segmentation_config)

def show_table(columns, rows):
    """Render concise text/Markdown tables; no visualization dependencies."""
    lines = ["| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines.extend("| " + " | ".join(str(value).replace("|", "/") for value in row) + " |"
                 for row in rows)
    text = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
    else:
        display(Markdown(text))

required = ["frames.jsonl", "references.jsonl", "metadata/dataset.json"]
show_table(["Input", "Available"], [(relative, (RUN_DIR / relative).is_file()) for relative in required])
print({"configured_sam": sam_settings,
       "proposed_vlm_budget_decimal_GB": 6, "budget_validation": "unmeasured",
       "sam_memory": "additional; unmeasured", "downloads_enabled": ALLOW_DOWNLOADS})


## Freeze the prepared selection

`PREPARE_SELECTION` only freezes already prepared inputs; it does not prepare datasets or download images. Missing source/split coverage stays incomplete. The public `freeze_selection` resolves the data root through the existing environment variable, restored after this call. A changed frozen image, reference, or policy requires a new run.


In [ ]:
if PREPARE_SELECTION:
    from contextlib import contextmanager
    from traversability_hazard_segmentation import freeze_selection

    @contextmanager
    def selection_data_root():
        previous = os.environ.get("TRAVERSABILITY_DATA_ROOT")
        os.environ["TRAVERSABILITY_DATA_ROOT"] = str(DATA_ROOT)
        try:
            yield
        finally:
            if previous is None:
                os.environ.pop("TRAVERSABILITY_DATA_ROOT", None)
            else:
                os.environ["TRAVERSABILITY_DATA_ROOT"] = previous

    with selection_data_root():
        selection = freeze_selection(RUN_DIR, test_frames_per_source=10, warmup_frames=5, seed=0)
    print({"complete": selection["complete"], "fixture": selection["fixture"],
           "test_frames": len(selection["test_frame_ids"]),
           "warmup_frames": len(selection["warmup_frame_ids"]),
           "fingerprint": selection["fingerprint"], "coverage": selection["coverage"]})
else:
    print("Selection preparation disabled.")


## Explicit CPU fixture

`RUN_FIXTURE` creates a uniquely named synthetic data/run pair, runs all four conditions, and measures fixture VLM and SAM calls separately. `ENABLE_COMBINED` additionally exercises the fixture combined path. These timings and null GPU memory are software checks, never real-model evidence. Fixtures never write to the selected real run.


In [ ]:
if RUN_FIXTURE:
    from uuid import uuid4
    from traversability_hazard_segmentation import run_conditions
    from traversability_hazard_segmentation.fixtures import prepare_fixture, FakeSegmenter, FakeBackend
    from traversability_hazard_benchmark import profile_backend

    fixture_tag = "hazard_segmentation_fixture_" + uuid4().hex[:12]
    fixture_run_dir = RUN_ROOT / fixture_tag
    fixture_data_root = DATA_ROOT / "fixtures" / fixture_tag
    fixture_config = prepare_fixture(fixture_run_dir, fixture_data_root)
    fixture_segmenter = FakeSegmenter()
    try:
        fixture_condition_summary = run_conditions(fixture_config, segmenter=fixture_segmenter)
    finally:
        fixture_segmenter.close()
    fixture_profiles = []
    for component, condition in (("vlm", "vlm__qwen3_vl_4b"), ("sam", "fixed_policy")):
        profile_config = deepcopy(benchmark_config)
        profile_config.update(fixture_config)
        profile_config.update(component=component, condition_key=condition,
                              model_key="qwen3_vl_4b", profile_id="cpu_fixture_" + component,
                              fixture=True, enable_real_run=False, enable_combined=False)
        owned_backend = FakeBackend("qwen3_vl_4b") if component == "vlm" else None
        owned_segmenter = FakeSegmenter() if component == "sam" else None
        if owned_backend is not None:
            profile_config["vlm_settings"] = deepcopy(owned_backend.settings)
        if owned_segmenter is not None:
            profile_config["sam_settings"] = deepcopy(owned_segmenter.settings)
        try:
            fixture_profiles.append(profile_backend(profile_config, backend=owned_backend,
                                                    segmenter=owned_segmenter))
        finally:
            if owned_backend is not None:
                owned_backend.close()
            if owned_segmenter is not None:
                owned_segmenter.close()
    if ENABLE_COMBINED:
        profile_config = deepcopy(benchmark_config)
        profile_config.update(fixture_config)
        profile_config.update(component="combined", condition_key="vlm__qwen3_vl_4b",
                              model_key="qwen3_vl_4b", profile_id="cpu_fixture_combined",
                              fixture=True, enable_real_run=False, enable_combined=True)
        owned_backend, owned_segmenter = FakeBackend("qwen3_vl_4b"), FakeSegmenter()
        profile_config.update(vlm_settings=deepcopy(owned_backend.settings),
                              sam_settings=deepcopy(owned_segmenter.settings))
        try:
            fixture_profiles.append(profile_backend(profile_config, backend=owned_backend,
                                                    segmenter=owned_segmenter))
        finally:
            owned_backend.close()
            owned_segmenter.close()
    print({"fixture_run": str(fixture_run_dir), "fixture": True,
           "condition_summary": fixture_condition_summary,
           "comparison_ready": False, "profiles": len(fixture_profiles)})
else:
    print("CPU fixture disabled.")


## Real segmentation

Enable `RUN_SEGMENTATION` only after completing checkpoint access and environment setup. `run_conditions` freezes/revalidates selection before reading predictions and owns its SAM resources. Missing VLM predictions remain incomplete; upstream/query errors preserve failures and partial masks. Successful empty lists and no detections have original-size empty masks.

For direct adapter use, the public API is `load_segmenter(settings)`, `segmenter.segment_image(frame, prompts, data_root)`, then `segmenter.close()` in a `finally` block. References are controller-only and never passed to the adapter. An externally injected adapter is allowed only in explicit fixture mode.


In [ ]:
if RUN_SEGMENTATION:
    from traversability_hazard_segmentation import run_conditions

    # The runner owns its real segmenter. Injected segmenters are fixture-only.
    condition_summary = run_conditions(segmentation_config)
    print(condition_summary)
else:
    print("Real SAM loading and segmentation disabled.")


## Real isolated and optional combined profiling

`RUN_PROFILES` measures whole-RGB VLM and SAM calls independently: five fixed development warmups and two repeats over the 20 test frames. Loading is separate. Each attempt includes synchronized elapsed time, errors, actual query counts, and available memory baselines/peaks. Input hash/dimension checks occur outside timing.

Both models reside together only when `ENABLE_COMBINED` is also enabled. That additional measured run provides joint peaks and combined latency; separate peaks and p95 values cannot establish joint GPU fit. The runner closes only resources it loads and never clears process-wide CUDA caches.


In [ ]:
if RUN_PROFILES:
    from traversability_hazard_benchmark import profile_backend

    profiles = []
    for component in ("vlm", "sam"):
        profile_config = deepcopy(benchmark_config)
        condition = (PROFILE_CONDITION_KEY or "vlm__" + PROFILE_MODEL_KEY) if component == "sam" else "vlm__" + PROFILE_MODEL_KEY
        profile_config.update(component=component, condition_key=condition,
                              model_key=PROFILE_MODEL_KEY, profile_id="isolated_" + component,
                              enable_combined=False)
        profiles.append(profile_backend(profile_config))
    if ENABLE_COMBINED:
        profile_config = deepcopy(benchmark_config)
        profile_config.update(component="combined", condition_key="vlm__" + PROFILE_MODEL_KEY,
                              model_key=PROFILE_MODEL_KEY, profile_id="combined",
                              enable_combined=True)
        profiles.append(profile_backend(profile_config))
else:
    print("Real profiling disabled; combined profiling needs both RUN_PROFILES and ENABLE_COMBINED.")


## Saved artifact tables

Tables report saved attempts and their fixture provenance. Canonical scoring concepts are bookkeeping; SAM receives the original phrases. Instance masks, each query union, and the all-query frame union retain original RGB dimensions. No visualization or legacy region profiling is implemented here.


In [ ]:
# Read-only tables. CPU fixture outputs have a separate run and remain marked.
table_run_dir = fixture_run_dir if RUN_FIXTURE and fixture_run_dir is not None else RUN_DIR
condition_rows = []
for condition in ("vlm__qwen3_vl_4b", "vlm__qwen3_5_4b", "reference_present", "fixed_policy"):
    path = table_run_dir / "segmentation" / condition / "frames.jsonl"
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()] if path.is_file() else []
    condition_rows.append((condition, len(records),
                           sum(row.get("status") == "ok" for row in records),
                           sum(row.get("status") == "error" for row in records),
                           sum(row.get("sam_query_count", 0) for row in records)))
show_table(["Condition", "Stored frames", "OK", "Errors", "SAM queries"], condition_rows)
profile_rows = []
profile_root = table_run_dir / "benchmark"
for path in sorted(profile_root.glob("*/*/summary.json")) if profile_root.is_dir() else []:
    summary = json.loads(path.read_text(encoding="utf-8"))
    latency, memory = summary.get("latency", {}), summary.get("memory", {})
    profile_rows.append((summary.get("component"), summary.get("condition_key"),
                         summary.get("fixture"), summary.get("measured_frames"),
                         summary.get("failed_calls"), latency.get("median_s"),
                         latency.get("p95_s"), memory.get("allocated_peak_bytes"),
                         summary.get("comparison_ready")))
show_table(["Stage", "Condition", "Fixture", "Attempts", "Failures", "Median s", "p95 s",
            "Allocated peak bytes", "Comparison ready"], profile_rows)
print("Null telemetry is unavailable. Joint peaks and combined p95 require an actual combined run.")
